<a href="https://colab.research.google.com/github/Santosdevbjj/identificar-habilidades-com-IA/blob/main/notebooks/01-skill-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import json

# Estrutura JSON sanitizada para o Colab
notebook_data = {
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🧠 Skill2Income AI - Análise Exploratória e Tokenização (EDA)\n",
    "\n",
    "Este notebook executa o passo fundamental de engenharia de dados do framework: a ingestão de perfis textuais brutos, higienização de strings e validação volumétrica de tokens utilizando o codificador oficial da OpenAI (`tiktoken`).\n",
    "\n",
    "**Objetivos:**\n",
    "1. Carregar a massa de dados bruta simulada (`data/raw/user_input_sample.json`).\n",
    "2. Higienizar e sanitizar os fluxos de texto (Remoção de quebras de linha e ruídos).\n",
    "3. Avaliar a volumetria de tokens para garantir o encaixe na janela de contexto (*Context Window*) e estimar custos operacionais da API."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Instalação das dependências necessárias para análise local\n",
    "!pip install pandas tiktoken"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "import os\n",
    "import pandas as pd\n",
    "import tiktoken"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Ingestão e Leitura dos Dados Brutos"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Navega a partir da pasta de notebooks para a raiz de dados\n",
    "raw_data_path = \"../data/raw/user_input_sample.json\"\n",
    "\n",
    "if not os.path.exists(raw_data_path):\n",
    "    print(\"⚠️ Nota: Para executar esta célula, lembre-se de que o arquivo JSON de dados brutos deve estar na pasta data/raw/ do seu projeto.\")\n",
    "else:\n",
    "    with open(raw_data_path, \"r\", encoding=\"utf-8\") as file:\n",
    "        payload_bruto = json.load(file)\n",
    "    print(f\"Usuário Identificado: {payload_bruto['usuario_id']}\")\n",
    "    print(f\"Fonte dos Dados: {payload_bruto['fonte_dados']}\")\n",
    "    print('-' * 50)\n",
    "    print('Amostra do Texto Capturado:')\n",
    "    print(payload_bruto['raw_text'][:350] + '...')\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Higienização e Tratamento do Texto\n",
    "\n",
    "Removemos quebras de linha duplicadas, espaços residuais e caracteres ocultos comuns em extrações de currículos em formato PDF."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "def limpar_texto_perfil(texto: str) -> str:\n",
    "    if not texto:\n",
    "        return \"\"\n",
    "    texto_limpo = texto.replace(\"\\n\", \" \").replace(\"\\r\", \" \")\n",
    "    texto_limpo = \" \".join(texto_limpo.split())\n",
    "    return texto_limpo\n",
    "\n",
    "try:\n",
    "    texto_processado = limpar_texto_perfil(payload_bruto[\"raw_text\"])\n",
    "    print(f\"Comprimento do texto bruto: {len(payload_bruto['raw_text'])} caracteres\")\n",
    "    print(f\"Comprimento do texto limpo: {len(texto_processado)} caracteres\")\n",
    "except NameError:\n",
    "    print(\"Definição de payload ausente. Execute apenas após carregar o arquivo de dados bruto.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Análise Volumétrica de Tokens (Metrificação OpenAI)\n",
    "\n",
    "Modelos como o `gpt-4o-mini` utilizam a codificação **cl100k_base** ou **o200k_base**. Vamos medir o volume exato de tokens consumido pelo perfil para garantir eficiência de custo e performance de rede."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "metadata": {},
   "outputs": [],
   "source": [
    "try:\n",
    "    encoder = tiktoken.get_encoding(\"cl100k_base\")\n",
    "    tokens_brutos = encoder.encode(payload_bruto[\"raw_text\"])\n",
    "    tokens_limpos = encoder.encode(texto_processado)\n",
    "    print(f\"Total de Tokens (Texto Bruto): {len(tokens_brutos)}\")\n",
    "    print(f\"Total de Tokens (Texto Limpo): {len(tokens_limpos)}\")\n",
    "    custo_estimado = (len(tokens_limpos) / 1_000_000) * 0.15\n",
    "    print(f\"Custo de Ingestão da Célula de IA: ${custo_estimado:.7f} USD\")\n",
    "except NameError:\n",
    "    print(\"Execute a célula 1 com sucesso primeiro.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Conclusão da Análise\n",
    "\n",
    "O perfil possui volumetria controlada (~400 a 600 tokens), o que se enquadra perfeitamente na janela de contexto padrão da API da OpenAI. Os dados limpos estão prontos para serem encaminhados de forma segura à esteira do `SkillExtractor` no microsserviço principal."
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "notebook_format": 4,
 "notebook_format_minor": 2
}

# Salva o arquivo fisicamente na máquina virtual do Colab
with open("01-skill-analysis.ipynb", "w", encoding="utf-8") as f:
    json.dump(notebook_data, f, indent=1, ensure_ascii=False)

print("✅ O notebook '01-skill-analysis.ipynb' foi gerado com sucesso!")

✅ O notebook '01-skill-analysis.ipynb' foi gerado com sucesso!
